# 03 LLM 推理机制：prefill、decode、KV cache 与调度

![LLM inference pipeline](../assets/images/llm_inference_pipeline_unlabeled.png)

本章讲所有 serving 框架都绕不开的机制。vLLM、SGLang、TensorRT-LLM、TGI 的实现不同，但都要处理同一组问题：prompt 如何 prefill，输出如何 decode，KV cache 如何存，多个请求如何 batch，长 prompt 如何不拖垮短请求。


## 1. Prefill 与 decode

```mermaid
sequenceDiagram
    participant C as 客户端 / Client
    participant S as 推理服务引擎 / Serving Engine
    participant M as 模型 / Model
    C->>S: prompt tokens
    S->>M: prefill(prompt tokens)
    M-->>S: KV cache + first logits
    S-->>C: first token (TTFT)
    loop every output token
        S->>M: decode(last token, KV cache)
        M-->>S: next logits
        S-->>C: stream token
    end
```

- **Prefill**：一次性处理 prompt 中所有 token，建立 KV cache。长 prompt 的 prefill 成本高。
- **Decode**：每一步生成一个 token，读取历史 KV cache。长输出和高并发会让 decode 压力变大。

指标关系：

$$
TTFT = t_{first\ token} - t_{request\ start}
$$

$$
ITL_i = t_{token_i} - t_{token_{i-1}}
$$

$$
TPS = \frac{\text{output tokens}}{\text{generation time}}
$$


In [ ]:
from dataclasses import dataclass

@dataclass
class Request:
    name: str
    prompt_tokens: int
    output_tokens: int

# 只是直觉模型，不代表真实硬件性能。
PREFILL_MS_PER_TOKEN = 0.035
DECODE_MS_PER_TOKEN = 7.0
QUEUE_MS = 25

requests = [
    Request('short_chat', 300, 80),
    Request('long_rag', 6000, 160),
    Request('agent_trace', 1800, 220),
]

for r in requests:
    ttft = QUEUE_MS + r.prompt_tokens * PREFILL_MS_PER_TOKEN + DECODE_MS_PER_TOKEN
    total = ttft + max(0, r.output_tokens - 1) * DECODE_MS_PER_TOKEN
    print(f'{r.name:12s} prompt={r.prompt_tokens:5d} output={r.output_tokens:3d} TTFT≈{ttft:7.1f}ms total≈{total:7.1f}ms')


## 2. KV cache 为什么关键

Transformer decode 时，新 token 需要关注所有历史 token。如果每一步都重新计算历史 token 的 Key/Value，会非常浪费。因此推理系统会保存历史 token 的 K/V，这就是 KV cache。

```mermaid
flowchart LR
    Prompt["提示词 tokens / prompt tokens"] --> Prefill["预填充 / prefill"]
    Prefill --> KV["KV 缓存 / KV cache<br/>每层 K/V"]
    KV --> D1["解码 token 1 / decode token 1"]
    D1 --> KV
    KV --> D2["解码 token 2 / decode token 2"]
    D2 --> KV
    KV --> D3["解码 token 3 / decode token 3"]
```

KV cache 是空间换时间：它减少重复计算，但消耗显存。上下文越长、并发越高，KV cache 越大。


## 3. Continuous batching

传统批处理可能等一批请求一起开始、一起结束。LLM 服务不适合这样，因为每个请求 prompt 和 output 长度不同。Continuous batching 的思路是：每个调度 step 动态把可运行请求组成 batch，让 GPU 尽量持续有活。

```mermaid
flowchart TB
    Q["等待队列 / waiting queue"] --> S["调度器 / scheduler"]
    R1["运行中请求 A / running req A"] --> S
    R2["运行中请求 B / running req B"] --> S
    S --> Step["一次引擎步 / one engine step<br/>混合 prefill/decode tokens"]
    Step --> Done["完成请求离开 / finished requests leave"]
    Step --> R1
    Step --> R2
    Q --> R3["新请求稍后加入 / new requests join later"]
```

它提升吞吐，但也引入调度权衡：大 prefill 可能阻塞 decode，导致流式输出卡顿；过度偏向 decode 又可能让新请求 TTFT 变差。


In [ ]:
# 一个极简 scheduler 模拟：比较 FCFS 和 chunked prefill 的直觉。
requests = [
    {'name': 'A_short', 'prefill': 400, 'decode': 80},
    {'name': 'B_long', 'prefill': 6000, 'decode': 80},
    {'name': 'C_short', 'prefill': 300, 'decode': 80},
]

def fcfs_ttft(reqs, prefill_rate=2000, decode_step_ms=8):
    now = 0
    result = {}
    for r in reqs:
        now += r['prefill'] / prefill_rate * 1000
        now += decode_step_ms
        result[r['name']] = now
    return result

def chunked_ttft(reqs, chunk=1000, prefill_rate=2000, decode_step_ms=8):
    now = 0
    remaining = {r['name']: r['prefill'] for r in reqs}
    result = {}
    while remaining:
        for r in reqs:
            name = r['name']
            if name not in remaining:
                continue
            take = min(chunk, remaining[name])
            now += take / prefill_rate * 1000
            remaining[name] -= take
            if remaining[name] == 0:
                now += decode_step_ms
                result[name] = now
                del remaining[name]
    return result

print('FCFS TTFT:', fcfs_ttft(requests))
print('Chunked prefill TTFT:', chunked_ttft(requests))


## 4. Prefix caching 与 RadixAttention

很多 Agent/RAG 请求共享长前缀：system prompt、工具说明、输出格式要求、企业政策模板。如果每次都重新 prefill 这些相同前缀，就浪费计算。Prefix caching 会复用相同前缀的 KV cache。

SGLang 的 RadixAttention 可以理解为更系统地管理共享前缀：把请求前缀组织成类似 radix tree 的结构，复用公共路径。vLLM 也支持 automatic prefix caching，核心目标都是减少重复 prefill。

```mermaid
flowchart TB
    Root["共享系统提示 / shared system prompt"] --> Tool["工具说明 / tool instructions"]
    Tool --> U1["用户问题 A / user question A"]
    Tool --> U2["用户问题 B / user question B"]
    Tool --> U3["用户问题 C / user question C"]
    Root --> Cache["前缀 KV 缓存复用 / prefix KV cache reused"]
    Tool --> Cache
```

prefix caching 的收益取决于：共享前缀长度、重复频率、cache 命中率、内存压力和 eviction 策略。


## 5. Speculative decoding

Speculative decoding 的直觉是：用一个更小更快的 draft model 先猜多个 token，再让大模型验证。猜对时可以一次接受多个 token，减少大模型 decode 步数；猜错时回退。

它适合 decode 瓶颈明显的场景，但不是万能：需要额外 draft model、验证逻辑和显存/算力预算。产品判断要看端到端延迟是否真的下降，而不是只看理论 token/sec。


## 6. 常见误区

- **TTFT 和 ITL 混为一谈。** TTFT 高常见于排队或 prefill；ITL 高常见于 decode、KV cache、后端 kernel 或负载问题。
- **只优化平均延迟。** AI 服务更要看 p95/p99，因为少数长请求会显著影响体验。
- **上下文越大越好。** 大上下文会增加 KV cache 和 prefill 成本，RAG 中应先优化检索和压缩。
- **batch 越大越好。** 大 batch 提升吞吐，但可能牺牲流式体验。


## 7. 从指标反推瓶颈

| 现象 | 可能原因 | 优先检查 |
|---|---|---|
| TTFT 高，ITL 正常 | 排队、prefill 慢、prompt 太长 | queue time、prompt tokens、prefill duration |
| TTFT 正常，ITL 高 | decode 慢、batch 过大、KV cache 压力 | time per output token、KV usage、GPU bandwidth |
| 平均正常，p99 很差 | 少数超长请求、突发并发、fallback | token 分布、租户切分、长尾请求 trace |
| OOM/preemption | KV cache 满、max_model_len 太大 | KV cache usage、并发、上下文限制 |
| 成本突然升高 | fallback、输出过长、prompt 膨胀 | model route、usage tokens、max_tokens |

这张表在排障时很实用：先确定是哪类指标异常，再回到具体层排查。


In [ ]:
# 从模拟请求中找出最可能的瓶颈类别。
records = [
    {'id': 'a', 'ttft': 0.8, 'itl': 0.04, 'queue': 0.1, 'prompt_tokens': 500},
    {'id': 'b', 'ttft': 4.2, 'itl': 0.05, 'queue': 2.8, 'prompt_tokens': 900},
    {'id': 'c', 'ttft': 3.8, 'itl': 0.05, 'queue': 0.2, 'prompt_tokens': 9000},
    {'id': 'd', 'ttft': 1.0, 'itl': 0.18, 'queue': 0.1, 'prompt_tokens': 700},
]
for r in records:
    if r['queue'] > 1.0:
        reason = 'queue bottleneck'
    elif r['ttft'] > 2.0 and r['prompt_tokens'] > 4000:
        reason = 'prefill / long prompt'
    elif r['itl'] > 0.1:
        reason = 'decode bottleneck'
    else:
        reason = 'healthy-ish'
    print(r['id'], reason)


## 8. 为什么 serving 框架都有 scheduler

没有 scheduler 的 naive 实现会让每个请求独占模型直到完成，这会浪费 GPU，也会让短请求被长请求拖死。Scheduler 要做的是在每个 engine step 选择一组 token 工作：哪些请求做 prefill，哪些请求做 decode，哪些请求等待，哪些请求因为内存压力被抢占或拒绝。

这和 Web 服务线程池类似，但更复杂，因为 LLM 请求的资源不是“一个连接”，而是 token 数、KV cache blocks、GPU step 时间和输出长度。调度策略直接影响用户体验和成本。


## 9. 一个长请求如何影响其他用户

LLM serving 的困难在于请求之间不是独立的。一个超长 RAG prompt 可能占用大量 prefill token budget；一个长输出 Agent 可能长期占据 decode step；多个长上下文请求会挤占 KV cache。即使每个请求单独看都合理，混在一起就可能让整体 p95/p99 变差。

这也是为什么生产系统常做请求分级：限制最大 prompt tokens、限制最大 output tokens、给交互式请求更高优先级、把离线批处理放到独立队列、对超长请求做异步任务或降级。服务质量不是只靠更强 GPU，也靠合理的 workload 管理。


## 10. Prefix caching 在 Agent/RAG 中的真实价值

Agent/RAG 请求常有大量重复前缀：系统角色、工具 schema、输出格式、合规要求、公司背景说明。如果每次请求都重新 prefill，浪费会非常明显。Prefix caching 的理想场景是：共享前缀长、重复频率高、用户差异部分短。

但 prefix caching 也不是完全免费。它需要占用 cache 空间；多租户场景要考虑隔离；prompt 中如果插入时间戳、随机 request id 或动态字段，可能破坏前缀命中。工程上要把稳定前缀放前面，把动态内容放后面，才能提高命中率。


## 11. 自测题

1. 为什么长 prompt 主要影响 TTFT，而长 output 主要影响总耗时和 decode 资源？
2. continuous batching 为什么能提高吞吐？它可能牺牲什么？
3. chunked prefill 解决什么问题？
4. prefix caching 为什么对 Agent 工具说明特别有用？
5. speculative decoding 为什么不是所有场景都适合？


## 12. 推理机制和用户体验的连接

用户不会说“prefill 太慢”或“KV cache pressure 太高”，用户只会说“怎么还没开始回答”“回答一顿一顿的”“有时候特别慢”。AI 产品/研发要把这些体验翻译成系统指标：没开始回答对应 TTFT，输出卡顿对应 ITL，偶发很慢对应 p95/p99，长时间无响应可能是 queue 或 timeout。

这也是为什么聊天产品常用 streaming。即使总耗时没有减少，只要 TTFT 下降，用户体感就会更好。但 streaming 也不是万能：如果 ITL 很差，用户会看到断断续续输出；如果输出很长，仍然要控制 max_tokens 和停止条件；如果后端排队严重，首 token 仍然慢。

因此一个成熟的推理体验优化，不是只追求 tokens/sec，而是同时优化 TTFT、ITL、总延迟、失败率和成本。


## 来源与延伸阅读

本章正文已经把关键内容消化进教程，下面链接只用于延伸阅读和核对最新细节：vLLM、vLLM-Metal、Ray Serve LLM、SGLang、TensorRT-LLM、Text Generation Inference、LiteLLM、KServe、NVIDIA GPU Operator、OpenTelemetry GenAI。
